In [0]:
# Let’s now perform the simple task of creating a range of numbers. This range of numbers is just
# like a named column in a spreadsheet:

myRange = spark.range(1000).toDF("number")

In [0]:
myRange.show(5)
myRange.count()

In [0]:
divisBy2 = myRange.where("number % 2 = 0")
divisBy2.show(5)

# An action instructs Spark to compute a result from a series of transformations.
# The simplest action is count, which gives us the total number of records in the DataFrame:
divisBy2.count()

In [0]:
flightData2015 = spark\
    .read\
    .option("inferSchema", "true")\
    .option("header", "true")\
    .csv("/Volumes/workspace/default/spark_definite_guide/flight-data/csv/2015-summary.csv")

In [0]:
flightData2015.take(3)

In [0]:
flightData2015.sort("count").explain()

In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "8")
flightData2015.sort("count").take(4)

In [0]:
flightData2015.createOrReplaceTempView("flight_data_2015")

# Now we can query our data in SQL. To do so, we’ll use the spark.sql function (remember,
# spark is our SparkSession variable) that conveniently returns a new DataFram

In [0]:
spark.sql("SELECT * FROM flight_data_2015 LIMIT 1").show()

In [0]:
spark.sql("DESCRIBE flight_data_2015").show()

In [0]:
sqlway = spark.sql("""
select dest_country_name, count(1)
from flight_data_2015
group by DEST_COUNTRY_NAME
""")


In [0]:
dataFrameway = flightData2015\
    .groupBy("DEST_COUNTRY_NAME")\
    .count()

In [0]:
sqlway.explain()

In [0]:
dataFrameway.explain()

In [0]:
spark.sql("SELECT max(count) from flight_data_2015").take(1)

# df = spark.sql("SELECT * FROM flight_data_2015")

# df.take(2)     # [Row(...), Row(...)]
# df.first()     # Row(...)
# df.show(2)     # prints 2 rows

In [0]:
maxSql = spark.sql("""
SELECT DEST_COUNTRY_NAME, sum(count) as destination_total
FROM flight_data_2015
GROUP BY DEST_COUNTRY_NAME
ORDER BY sum(count) DESC
LIMIT 5
""")

maxSql.show()


In [0]:
# Now, let’s move to the DataFrame syntax that is semantically similar but slightly different in
# implementation and ordering. But, as we mentioned, the underlying plans for both of them are
# the same. Let’s run the queries and see their results as a sanity check:

# DataFrame version
from pyspark.sql.functions import desc

flightData2015\
    .groupBy("DEST_COUNTRY_NAME")\
    .sum("count")\
    .withColumnRenamed("sum(count)", "destination_total")\
    .sort(desc("destination_total"))\
    .limit(5)\
    .show()

